# Module 4: Multimodal Fusion Network — Training

Trains the FoodIntel FNN that fuses CNN visual embeddings (v ∈ R^512) and
NLP text embeddings (t ∈ R^512) via cross-attention, then predicts:

| Head | Outputs | Loss |
|------|---------|------|
| Nutrition Regression | calories, protein, fat, carbs | MSE |
| Health Risk Classification | cardiovascular, diabetes, hypertension | CrossEntropy |
| Dietary Compatibility | vegan, keto, high-protein, gluten-free | BCEWithLogits |

#### Imports & Config

In [ ]:
import os
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, random_split
import matplotlib.pyplot as plt

from fusion_model import (
    FoodIntelFNN, LossWeights,
    VISUAL_DIM, TEXT_DIM, FUSED_DIM,
    NLP_CONSTRAINT_ROWS, NLP_CONSTRAINT_COLS,
    NUM_NUTRITION_TARGETS, NUM_RISK_CLASSES, NUM_DIET_TAGS,
    NUTRITION_NAMES, RISK_NAMES, DIET_NAMES,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Training configuration ───────────────────────────────────────────
EPOCHS        = 30
BATCH_SIZE    = 32
LR_FUSION     = 1e-4     # fusion layer LR
LR_HEADS      = 1e-3     # prediction heads LR (10x higher)
WEIGHT_DECAY  = 0.01
MAX_GRAD_NORM = 1.0
VAL_SPLIT     = 0.2
USE_CONSTRAINT = False   # toggle NLP 5x12 constraint injection
DROPOUT       = 0.1

# Multi-task loss weights (equal by default)
LOSS_WEIGHTS  = LossWeights(nutrition=1.0, risk=1.0, diet=1.0)

# Output
CKPT_DIR = Path('checkpoints/fusion')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Data mode: set USE_SYNTHETIC=False and DATA_DIR to use real embeddings
USE_SYNTHETIC  = True
SYNTHETIC_SIZE = 2000
DATA_DIR       = 'outputs/fusion_data'  # only used when USE_SYNTHETIC=False

#### Datasets

In [ ]:
class SyntheticFusionDataset(Dataset):
    """Random L2-normalised embeddings for architecture smoke-testing."""
    def __init__(self, size=2000, seed=42):
        rng = np.random.RandomState(seed)
        v = rng.randn(size, VISUAL_DIM).astype(np.float32)
        self.visual = v / np.linalg.norm(v, axis=1, keepdims=True)
        t = rng.randn(size, TEXT_DIM).astype(np.float32)
        self.text = t / np.linalg.norm(t, axis=1, keepdims=True)
        self.constraint = rng.randn(size, NLP_CONSTRAINT_ROWS, NLP_CONSTRAINT_COLS).astype(np.float32)
        self.nutrition = rng.rand(size, NUM_NUTRITION_TARGETS).astype(np.float32)
        self.risk = rng.randint(0, NUM_RISK_CLASSES, size=size).astype(np.int64)
        self.diet = rng.randint(0, 2, size=(size, NUM_DIET_TAGS)).astype(np.float32)

    def __len__(self): return len(self.visual)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.visual[idx]),
            torch.from_numpy(self.text[idx]),
            torch.from_numpy(self.constraint[idx]),
            torch.from_numpy(self.nutrition[idx]),
            torch.tensor(self.risk[idx]),
            torch.from_numpy(self.diet[idx]),
        )


class FusionDataset(Dataset):
    """Load pre-computed CNN and NLP embeddings from .npz files.

    Expected layout:
      data_dir/visual_embeddings.npz  — key: embedding (N, 512)
      data_dir/nlp_embeddings.npz     — keys: embedding (N, 512), constraint (N, 5, 12)
      data_dir/targets.npz            — keys: nutrition (N, 4), risk (N,), diet (N, 4)
    """
    def __init__(self, data_dir):
        d = Path(data_dir)
        vis = np.load(d / 'visual_embeddings.npz')
        nlp = np.load(d / 'nlp_embeddings.npz')
        tgt = np.load(d / 'targets.npz')
        self.visual     = vis['embedding'].astype(np.float32)
        self.text       = nlp['embedding'].astype(np.float32)
        self.constraint = nlp['constraint'].astype(np.float32)
        self.nutrition  = tgt['nutrition'].astype(np.float32)
        self.risk       = tgt['risk'].astype(np.int64)
        self.diet       = tgt['diet'].astype(np.float32)

    def __len__(self): return len(self.visual)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.visual[idx]),
            torch.from_numpy(self.text[idx]),
            torch.from_numpy(self.constraint[idx]),
            torch.from_numpy(self.nutrition[idx]),
            torch.tensor(self.risk[idx]),
            torch.from_numpy(self.diet[idx]),
        )

In [ ]:
# Build dataloaders
if USE_SYNTHETIC:
    full_ds = SyntheticFusionDataset(size=SYNTHETIC_SIZE)
    print(f'Synthetic dataset: {len(full_ds)} samples')
else:
    full_ds = FusionDataset(DATA_DIR)
    print(f'Real dataset from {DATA_DIR}: {len(full_ds)} samples')

val_size   = int(len(full_ds) * VAL_SPLIT)
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {train_size}  Val: {val_size}')

#### Model, Optimizer & Scheduler

In [ ]:
model = FoodIntelFNN(
    use_constraint=USE_CONSTRAINT,
    dropout=DROPOUT,
).to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {trainable_p:,} trainable / {total_p:,} total')

# Differential LR: fusion layer gets 10x lower LR than heads
fusion_params = list(model.fusion.parameters())
if model.constraint_injector is not None:
    fusion_params += list(model.constraint_injector.parameters())
head_params = (
    list(model.nutrition_head.parameters())
    + list(model.risk_head.parameters())
    + list(model.diet_head.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': fusion_params, 'lr': LR_FUSION},
    {'params': head_params,   'lr': LR_HEADS},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f'Optimizer: AdamW (fusion LR={LR_FUSION}, heads LR={LR_HEADS})')
print(f'Scheduler: CosineAnnealingLR (T_max={EPOCHS})')

#### Training & Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_weights, device):
    model.train()
    accum = {'total': 0., 'nutrition': 0., 'risk': 0., 'diet': 0.}
    n = 0
    for v, t, c, nut, risk, diet in loader:
        v, t, c   = v.to(device), t.to(device), c.to(device)
        nut, risk, diet = nut.to(device), risk.to(device), diet.to(device)

        outputs = model(v, t, constraint=c)
        total_loss, losses = model.compute_loss(
            outputs, {'nutrition': nut, 'risk': risk, 'diet': diet}, loss_weights)

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

        for k in accum: accum[k] += losses[k].item()
        n += 1
    return {k: v / max(n, 1) for k, v in accum.items()}


@torch.inference_mode()
def validate(model, loader, loss_weights, device):
    model.eval()
    accum = {'total': 0., 'nutrition': 0., 'risk': 0., 'diet': 0.}
    mae_sum, risk_ok, diet_ok, n_samples, n = 0., 0, 0., 0, 0

    for v, t, c, nut, risk, diet in loader:
        v, t, c   = v.to(device), t.to(device), c.to(device)
        nut, risk, diet = nut.to(device), risk.to(device), diet.to(device)

        outputs = model(v, t, constraint=c)
        _, losses = model.compute_loss(
            outputs, {'nutrition': nut, 'risk': risk, 'diet': diet}, loss_weights)

        for k in accum: accum[k] += losses[k].item()
        bs = v.size(0)
        mae_sum  += (outputs['nutrition'] - nut).abs().mean().item() * bs
        risk_ok  += (outputs['risk'].argmax(1) == risk).sum().item()
        diet_ok  += ((outputs['diet'] > 0).float() == diet).float().mean().item() * bs
        n_samples += bs
        n += 1

    m = {k: v / max(n, 1) for k, v in accum.items()}
    m['nutrition_mae']  = mae_sum  / max(n_samples, 1)
    m['risk_accuracy']  = risk_ok  / max(n_samples, 1)
    m['diet_accuracy']  = diet_ok  / max(n_samples, 1)
    return m

#### Training Loop

In [ ]:
history = {f'{s}_{t}': [] for s in ('train','val') for t in ('total','nutrition','risk','diet')}
history.update({'val_nutrition_mae': [], 'val_risk_accuracy': [], 'val_diet_accuracy': []})

best_val_loss = float('inf')
start = perf_counter()

for epoch in range(1, EPOCHS + 1):
    tm = train_one_epoch(model, train_loader, optimizer, LOSS_WEIGHTS, device)
    vm = validate(model, val_loader, LOSS_WEIGHTS, device)
    scheduler.step()

    for t in ('total','nutrition','risk','diet'):
        history[f'train_{t}'].append(tm[t])
        history[f'val_{t}'].append(vm[t])
    history['val_nutrition_mae'].append(vm['nutrition_mae'])
    history['val_risk_accuracy'].append(vm['risk_accuracy'])
    history['val_diet_accuracy'].append(vm['diet_accuracy'])

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Train {tm['total']:.4f} | Val {vm['total']:.4f} | "
          f"MAE {vm['nutrition_mae']:.4f} | "
          f"RiskAcc {vm['risk_accuracy']:.3f} | "
          f"DietAcc {vm['diet_accuracy']:.3f} | "
          f"LR {scheduler.get_last_lr()[0]:.2e}")

    if vm['total'] < best_val_loss:
        best_val_loss = vm['total']
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': best_val_loss,
        }, CKPT_DIR / 'fusion_best.pth')
        print(f'  -> Saved fusion_best.pth (val_loss={best_val_loss:.4f})')

elapsed = perf_counter() - start
print(f'\nDone in {elapsed:.1f}s — best val loss: {best_val_loss:.4f}')

#### Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history['train_total'], label='Train', marker='o', ms=3)
axes[0,0].plot(history['val_total'],   label='Val',   marker='o', ms=3)
axes[0,0].set_title('Total Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

for key, title, ax in [
    ('nutrition', 'Nutrition (MSE)', axes[0,1]),
    ('risk',      'Health Risk (CE)', axes[1,0]),
    ('diet',      'Diet Tags (BCE)',  axes[1,1]),
]:
    ax.plot(history[f'train_{key}'], label='Train', marker='o', ms=3)
    ax.plot(history[f'val_{key}'],   label='Val',   marker='o', ms=3)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('FoodIntel FNN \u2014 Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

#### Validation Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['val_nutrition_mae'], marker='o', ms=3, color='teal')
axes[0].set_title('Nutrition MAE'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_risk_accuracy'], marker='o', ms=3, color='coral')
axes[1].set_title('Risk Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(history['val_diet_accuracy'], marker='o', ms=3, color='mediumpurple')
axes[2].set_title('Diet Tag Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)

plt.suptitle('FoodIntel FNN \u2014 Validation Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CKPT_DIR / 'validation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved validation_metrics.png')

#### Shape Verification & Attention Weights

In [ ]:
# Load best checkpoint and verify shapes
ckpt = torch.load(CKPT_DIR / 'fusion_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded fusion_best.pth (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

# Forward pass with dummy data
dummy_v = torch.randn(4, VISUAL_DIM).to(device)
dummy_t = torch.randn(4, TEXT_DIM).to(device)
dummy_c = torch.randn(4, NLP_CONSTRAINT_ROWS, NLP_CONSTRAINT_COLS).to(device)

with torch.no_grad():
    out = model(dummy_v, dummy_t, constraint=dummy_c)

print(f'\nOutput shapes:')
print(f'  nutrition:         {out["nutrition"].shape}   (expected: [4, {NUM_NUTRITION_TARGETS}])')
print(f'  risk:              {out["risk"].shape}   (expected: [4, {NUM_RISK_CLASSES}])')
print(f'  diet:              {out["diet"].shape}   (expected: [4, {NUM_DIET_TAGS}])')
print(f'  fused:             {out["fused"].shape} (expected: [4, {FUSED_DIM}])')
print(f'  attention_weights: {out["attention_weights"].shape} (expected: [4, 2, 2])')

# Attention weight analysis
w = out['attention_weights'].cpu().numpy().mean(axis=0)  # average over batch
print(f'\nMean attention weights (visual, text):')
print(f'  Visual attends to: visual={w[0,0]:.3f}, text={w[0,1]:.3f}')
print(f'  Text   attends to: visual={w[1,0]:.3f}, text={w[1,1]:.3f}')

#### Save Manifest

In [ ]:
manifest = {
    'model': 'FoodIntelFNN',
    'modalities': ['visual (CNN -> 512-d)', 'text (NLP -> 512-d)'],
    'use_constraint': USE_CONSTRAINT,
    'fusion_dim': FUSED_DIM,
    'heads': {
        'nutrition': {'outputs': NUM_NUTRITION_TARGETS, 'loss': 'MSELoss',
                      'names': NUTRITION_NAMES},
        'risk':      {'outputs': NUM_RISK_CLASSES, 'loss': 'CrossEntropyLoss',
                      'names': RISK_NAMES},
        'diet':      {'outputs': NUM_DIET_TAGS, 'loss': 'BCEWithLogitsLoss',
                      'names': DIET_NAMES},
    },
    'training': {
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'optimizer': 'AdamW',
        'lr_fusion': LR_FUSION, 'lr_heads': LR_HEADS,
        'scheduler': 'CosineAnnealingLR',
        'best_val_loss': round(best_val_loss, 6),
        'elapsed_seconds': round(elapsed, 2),
    },
}
(CKPT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Saved manifest.json')
print(json.dumps(manifest, indent=2))